In [1]:
from dataclasses import dataclass
import sys
sys.path.append('../estimator')

from estimator import LWE, ND
from estimator.nd import sigmaf,stddevf
from sage.all import var, log, ceil, floor, sqrt, Infinity, euler_phi, round, pi, n, ln

In [2]:
import os

class HiddenPrints:
    def __enter__(self):
        self._original_stdout = sys.stdout
        sys.stdout = open(os.devnull, "w")

    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout.close()
        sys.stdout = self._original_stdout

In [3]:
def smoothing_par_gsbasis(*, gsmax, m, eps):
    # [LPR13, Eprint Lemma 2.6] and [GPV08, Eprint Lemma 3.1]
    return gsmax* sqrt(ln(2*m*(1 + 1/eps))/pi)

def smoothing_par_Zqn(*, q, n, eps):
    return smoothing_par_gsbasis(gsmax=1, m=n, eps=eps)

def aslog(x):
    try:
        y = float(log(x, 2))
        return f"2^{y:.1f}"
    except:
        return "NOT A FLOAT"

In [4]:
def lwe_estimate_security(params, rough=True):
    if rough:
        est = LWE.estimate.rough(params)
    else:
        est = LWE.estimate(params)
    return float(min([log(algo["rop"], 2) for algo in est.values()]))

def sis_estimate_security(params, rough=True):
    if rough:
        est = SIS.estimate.rough(params)
    else:
        est = SIS.estimate(params)
    return float(min([log(algo["rop"], 2) for algo in est.values()]))

In [5]:
@dataclass(kw_only=True)
class RBEParams:
    """
    Dataclass for the parameters of the RBE.
    """

    secpar: int # security level
    f: int # conductor
    n: int # module rank, height of matrix A 
    l_A: int # m_A = n * l_A
    k: int = 3 # arity
    # m_A: int # width of matrix A
    gbase: int # decomposition base
    sigma: float      # Gaussian parameter
    sigma_tilde: float # Gaussian parameter
    sigma_star: float # Gaussian parameter
    q: int # modulus
    D: int # number of bits dropped
    B: int # number of dictionaries
    l_hat: int = 162 # max identity length

    def __repr__(self):
        return f'RBE( secpar: {self.secpar:3d}, f: {self.f:3d}, n: {self.n:1d}, l_A: {self.l_A:1d}, k: {self.k:1d}, m_A: {self.m_A:1d}, gbase: {int(self.gbase)}, \
q: {int(round(self.q))} ≈ 2^{ceil(log(self.q,2)):.1f}, sigma: {aslog(self.sigma)}, sigma_tilde: {aslog(self.sigma_tilde)}, sigma_star: {aslog(self.sigma_star)}, sigma_bar: {aslog(self.sigma_bar)} \
D: {int(self.D)}, B={int(self.B)}, (|ct|: {self.ct_size_MB():.1f} MB) \
correctness_ratio: 2^{float(log(self.ratio_optimal_correctness(), 2)):.3f}, security_ratio: 2^{float(log(self.ratio_optimal_security(), 2)):.3f}'

    def as_init_string(self):
        # Output a string which can be used to generate (a copy of) this object.
        return f'RBEParams(secpar={self.secpar}, f={self.f}, n={self.n}, l_A={self.l_A}, k={self.k}, gbase={int(self.gbase)}, \
q={int(round(self.q))}, sigma=2**{float(log(self.sigma.n(),2)):.3f}, sigma_tilde=2**{float(log(self.sigma_tilde.n(),2)):3f}, sigma_star=2**{float(log(self.sigma_star,2)):3f}, \
D={int(self.D)}, B={int(self.B)})'
    
    @property
    def m_A(self):
        return self.n * self.l_A
    
    @property
    def varphi(self):
        return euler_phi(self.f)

    @property
    def sigma_bar(self):
        return sqrt(self.sigma_tilde**2 - self.sigma**2)

    def minimal_gbase(self):
        return ceil(self.q**(1/(self.l_A + 1)))
    
    def ct_size_bits(self):
        # Return ciphertext size in Bits
        # Number of Zq elements in c and d component of ciphertext. Zq elements encoded with D bits dropped.
        n_c = self.l_hat * self.k * self.m_A
        n_c_l = self.n
        n_d = self.B
        return int((n_c + n_d) * self.varphi * (ceil(log(self.q, 2)) - self.D))
    
    def ct_size_MB(self):
        return float(self.ct_size_bits() / 8_000_000)

    def error_term_1(self):
        # First error term: Linear in sigma
        return 4 * ((self.k * self.l_A + 1)*self.l_hat + 2) * self.varphi**(3/2) * self.n * self.sigma * self.gbase/2

    def error_term_2(self):
        # Second error term: Linear in sigma_tilde
        return 4 * sqrt(self.varphi) * self.sigma_tilde

    def error_term_3(self):
        # Third error term: Linear in D (bit-dropping)
        return 4 * (self.varphi) * 2**(self.D-1) * (1 + (self.l_hat + 1) * self.gbase/2)

    def balanced_D(self):
        return floor(1 + log((self.error_term_1() + self.error_term_2()) / (4 * (self.varphi) * (1 + (self.l_hat + 1) * self.gbase/2)), 2))


    def balanced_error_sigma_tilde(self):
        return self.error_term_1() / (4 * sqrt(self.varphi))

    def balanced_lwe_sigma_tilde(self):
        return ((self.k * self.l_A + 1)*self.l_hat + 2) * self.varphi**2 * self.n * self.sigma * self.gbase**2
    
    def error_term_total(self):
        # Terms in equation in Cor. 4.3
        s1 = self.error_term_1()
        s2 = self.error_term_2()
        s3 = self.error_term_3()
        q = floor(s1 + s2 + s3 + 2)
        return q

    def minimal_q(self):
        # Terms in equation in Cor. 4.3
        return max(self.error_term_total(),
                   ceil(self.gbase**(self.l_A+1)))
        
    
    def is_correct(self):
        try:
            assert(self.gbase >= self.minimal_gbase()), f"self.gbase >= self.minimal_gbase(): {aslog(self.gbase)} >= {aslog(self.minimal_gbase())}"       
            assert self.q >= self.error_term_total(), f"q too small: {aslog(self.error_term_total())} >= {aslog(self.q)}"
        except AssertionError as e:
            print(f'Assertion failed: {e}')
            return False    
        return True

    def minimal_sigma_tilde(self, *, scale = 1.01):
        # Compute minimal sigma_tilde through Cor. 4.6, using \chibar + \chibar = \chitilde.
        smpar = self.smoothing_param(self.varphi * ((1 + self.k * self.l_A)*self.n + self.B)) # Common upper bound on smoothing parameters occuring in Cor. 4.6
        term_rhs = 1/(2 * smpar**2 + self.sigma_star**2) - 1/(self.sigma**2)
        term_lhs_numerator = ((self.k * self.l_A + 1) * self.n * self.B * self.varphi**2 * self.gbase**2)
        # Compute 1/sigma_bar^2 first. Then invert.
        sigbar_sq_inv =  4 * term_rhs/term_lhs_numerator
        sigbar = sqrt(1/sigbar_sq_inv)
        # Derive sigma_tilde
        sigtilde = sqrt(2) * sigbar
        assert scale > 1
        return scale * sigtilde # Scaled to be stricly larger. It shouldn't matter how much, so keep it very small.

    def smoothing_param(self, m):
        return smoothing_par_Zqn(q = self.q, n = m, eps = 2**(-self.secpar))

    def max_smoothing_param(self):
        # Maximal smoothing parameter expression that occurs.
        return self.smoothing_param(self.varphi * (1 + self.k * self.l_A)*self.n + self.B)
    
    def minimal_sigma_star(self):
        return sqrt(2) * self.smoothing_param(self.varphi * (1 + self.k * self.l_A)*self.n + self.B) # Slightly bigger than the lower bound, due to "1 + " term.

    def maximal_sigma_star(self):
        term1 = ((self.k * self.l_A + 1) * self.n * self.B * self.varphi**2 * self.gbase**2) / (4 * self.sigma_bar)
        sigstar_sq = 1/(1/term1 + 1/self.sigma**2) - self.smoothing_param(self.varphi * (1 + self.k * self.l_A)*self.n + self.B)
        return sqrt(sigstar_sq)

    
    def lwe_hardness(self,*, rough=True):
        lwe1 = LWE.Parameters(n = self.varphi*self.n, 
                              m = self.varphi*self.n,
                              q=self.q,
                              Xs=ND.Uniform(-self.gbase/2, self.gbase/2),
                              Xe=ND.Uniform(-self.gbase/2, self.gbase/2))
        lwe2 = LWE.Parameters(n=self.varphi*self.n,
                              m = self.varphi * (self.k * self.n * self.l_A + self.B),
                              q=self.q,
                              Xs=ND.DiscreteGaussian(stddevf(self.sigma_star)),
                              Xe=ND.DiscreteGaussian(stddevf(self.sigma_star)))
        with HiddenPrints():
            sec1 = lwe_estimate_security(lwe1, rough=rough)
            sec2 = lwe_estimate_security(lwe2, rough=rough)
        return (sec1, sec2)
        
    
    def is_secure(self, rough=True):
        try:
            # Cor. 4.6
            # Basic lower bounds
            assert self.sigma >= sqrt(2) * self.smoothing_param(self.varphi * self.B), f"self.sigma too small: {aslog(self.sigma)} >= {aslog(sqrt(2) * self.smoothing_param(self.varphi * self.B))}"
            assert self.sigma_bar >= sqrt(2) * self.smoothing_param(self.varphi * self.B), f"self.sigma_bar too small: {aslog(self.sigma_bar)} >= {aslog(sqrt(2) * self.smoothing_param(self.varphi * self.B))}"
            assert self.sigma_star >= self.minimal_sigma_star(), f"self.sigma_star too small: {aslog(self.sigma_star)} >= {aslog(self.minimal_sigma_star)}"

            # Relation between Gaussian widths
            lhs = ((self.k * self.l_A + 1) * self.n * self.B * self.varphi**2 * self.gbase**2) / (4 * self.sigma_bar**2)
            rhs = 1/(2 * self.smoothing_param(self.varphi * (1 + self.k * self.l_A)*self.n + self.B)**2 + self.sigma_star**2) - 1/(self.sigma**2)
            assert lhs <= rhs, f"IND$-CPA relation on gaussian width fail: Factor of {float(lhs/rhs) :.3f}"

            # Hardness of LWE parameters
            (sec1, sec2) = self.lwe_hardness(rough=rough)
            #assert sec1 >= 128, f"sec1 = {sec1:.1f}"
            #assert sec2 >= 128, f"sec2 = {sec2:.1f}"
            print(f"sec1 = {sec1:.1f} \nsec2 = {sec2:.1f}")
        except AssertionError as e:
            print(f'Assertion failed: {e}')
            return False    
        return True

    def heuristic_sigma_tilde(self):
        return 3 * self.l_hat * self.varphi * self.l_A * self.n * self.sigma * self.gbase

    def heuristic_q(self):
        return 12 * self.l_hat * self.varphi**(3/2) * self.l_A * self.n *self.sigma * self.gbase

    def ratio_optimal_correctness(self):
        return ((3 * self.l_A +1) * self.l_hat + 2) * self.varphi * self.n * self.sigma * self.gbase / (2 * self.sigma_tilde)

    def ratio_optimal_security(self):
        return (self.sigma * sqrt((self.k * self.l_A + 1) * self.n * self.B) * self.varphi * self.gbase) / (sqrt(2) * self.sigma_bar)

# Parameters chosen in the paper

In [6]:
#print(pars.as_init_string())
pars = RBEParams(secpar=128, f=512, n=7, l_A=2, k=3, gbase=2**16, q=281_474_976_694_273, sigma=2**4, sigma_tilde=2**34.964301, sigma_star=2**3.750000, D=16, B=30)
print(f"Correct: {pars.is_correct()}")
print(f"Secure: {pars.is_secure(rough=False)}")
print(f"{pars}")

Correct: True
sec1 = 304.6 
sec2 = 134.4
Secure: True
RBE( secpar: 128, f: 512, n: 7, l_A: 2, k: 3, m_A: 14, gbase: 65536, q: 281474976694273 ≈ 2^48.0, sigma: 2^4.0, sigma_tilde: 2^35.0, sigma_star: 2^3.8, sigma_bar: 2^35.0 D: 16, B=30, (|ct|: 7.0 MB) correctness_ratio: 2^4.993, security_ratio: 2^-2.204


---

# Figuring out parameters

In [ ]:
pars = RBEParams(secpar = 128,
                 f = 512,
                 n = 7,
                 l_A = 2,
                 sigma = 2**4,
                 sigma_tilde = 2**20, # None,
                 sigma_star = 2**3.75,
                 gbase = None,
                 q = None,
                 D = None, #20, # None,
                 B = 30,
                 k = 3,
                 l_hat = 162
                )

# We need to provide some basis for gadget G to work with.
pars.gbase = floor(2**16)
# Use the minimal possible sigma_tilde (plus a small safety factor).
pars.sigma_tilde = pars.minimal_sigma_tilde()
# Derive a heuristic D for bit-dropping. (Heuristic: Make error term summands balanced. The (balanced) error terms associated with sigma and sigma_tilde serve as the anchor.)
pars.D = pars.balanced_D()
# Increase D (by trial an error), because minimizing sigma_tilde has resulted in an *imbalanced* error term now.
pars.D += 2

# Recompute q as the minimal choice which ensures correctness given sigma, sigma_tilde, D
pars.q = pars.minimal_q() # Too many parameters. need fixed-point iteration...
#pars.q = 281_474_976_694_273 # This is a nice NTT-friendly prime.
# Check and output the resulting parameters.

if pars.is_correct() pars.is_secure(rough=False):
    print(f"{pars}")
    print(pars.as_init_string())

In [ ]:
pars = RBEParams(secpar = 128,
                 f = 512,
                 n = 6,
                 l_A = 2,
                 sigma = 2**5.5,
                 sigma_tilde = 2**20, # None,
                 sigma_star = 2**5.4,
                 gbase = None,
                 q = None,
                 D = None, #20, # None,
                 B = 30,
                 k = 3,
                 l_hat = 162
                )

# We need to provide some basis for gadget G to work with.
pars.gbase = floor(2**16)
# Use the minimal possible sigma_tilde (plus a small safety factor).
pars.sigma_tilde = pars.minimal_sigma_tilde()
# Derive a heuristic D for bit-dropping. (Heuristic: Make error term summands balanced. The (balanced) error terms associated with sigma and sigma_tilde serve as the anchor.)
pars.D = pars.balanced_D()
# Increase D (by trial an error), because minimizing sigma_tilde has resulted in an *imbalanced* error term now.
pars.D += 0

# Recompute q as the minimal choice which ensures correctness given sigma, sigma_tilde, D
pars.q = pars.minimal_q() # Too many parameters. need fixed-point iteration...
#pars.q = 281_474_976_694_273 # This is a nice NTT-friendly prime.
# Check and output the resulting parameters.

if pars.is_correct() pars.is_secure(rough=False):
    print(f"{pars}")
    print(pars.as_init_string())

In [ ]:
# 123 bit security at 6.2 MB
pars = RBEParams(secpar=128, f=512, n=6, l_A=2, k=3, gbase=65536, q=281_474_976_694_273, sigma=2**5.500, sigma_tilde=2**35.760202, sigma_star=2**5.400000, D=15, B=30)
print(pars.is_correct())
print(pars.is_secure(rough=False))
print(f"{pars}")

In [ ]:
# Heuristic correctness of the above: Shaving off one bit.
# 126 bit security at 6.0MB
# (TODO: Use a NTT-friendly q instead of power-of-2
pars = RBEParams(secpar=128, f=512, n=6, l_A=2, k=3, gbase=65536, q=2**47, sigma=2**5.500, sigma_tilde=2**35.760202, sigma_star=2**5.400000, D=15, B=30)
print(pars.is_correct())
print(pars.is_secure(rough=False))
print(f"{pars}")

In [ ]:
# Conservative correctness with 122 bit security at 5.8 MB

pars = RBEParams(secpar = 128,
                 f = 256,
                 n = 12,
                 l_A = 2,
                 sigma = 2**4.0,
                 sigma_tilde = None,
                 sigma_star = 2**3.75,
                 gbase = None,
                 q = None,
                 D = None,
                 B = 60, # Doubled to 30, so that still 256bit payload.
                 k = 3,
                 l_hat = 162
                )

# We need to provide some basis for gadget G to work with.
pars.gbase = floor(2**15)
# Use the minimal possible sigma_tilde (plus a small safety factor).
pars.sigma_tilde = pars.minimal_sigma_tilde()
# Derive a heuristic D for bit-dropping. (Heuristic: Make error term summands balanced. The (balanced) error terms associated with sigma and sigma_tilde serve as the anchor.)
pars.D = pars.balanced_D()
# Increase D (by trial an error), because minimizing sigma_tilde has resulted in an *imbalanced* error term now.
pars.D += 0

# Recompute q as the minimal choice which ensures correctness given sigma, sigma_tilde, D
pars.q = pars.minimal_q() # Too many parameters. need fixed-point iteration...
#pars.q = 281_474_976_694_273 # This is a nice NTT-friendly prime.
# Check and output the resulting parameters.

if pars.is_correct() and pars.is_secure(rough=False):
    print(f"{pars}")
    print(pars.as_init_string())

# Heuristically reducing q

In [ ]:
# Let's continue with the above, but shrink q by a reasonable amount.
# Notes:
# - Shrinking q increases LWE hardness and reduces ciphertext size.
# - Shrinking q the minimal q also corresponds to assuming a smaller effective error (i.e., assuming a bit of slack in our correctness analysis).
# - Increasing D (bits dropped) results in an increased minimal_q(), because minimal_q() ensures correctness.
# - Increasing D without recomputing q does not increase LWE hardness, because the error distribution does not account for the "extra noise" introduced by D.

pars = RBEParams(secpar=128, f=256, n=12, l_A=2, k=3, gbase=32768, q=35184372088832, sigma=2**4.000, sigma_tilde=2**33.849312, sigma_star=2**3.750000, D=14, B=60)

# Shave off some k bits to see the effect.
# Ensure 2**k <= sqrt(pars.varphi) else the correctness error is likely too big.
pars.q = floor(pars.q / 2**2)

print(pars.is_correct())
print(pars.is_secure(rough=False))
print(f"{pars}")
print(pars.as_init_string())

In [ ]:
# Another set of conservative parameters, but with only 114 bit security.
pars = RBEParams(secpar=128, f=256, n=11, l_A=2, k=3, gbase=32768, q=35184372088832, sigma=2**4.000, sigma_tilde=2**33.849312, sigma_star=2**3.750000, D=14, B=60)

pars.gbase = floor(2**15)
pars.sigma = 2**4.3
pars.sigma_star = 2**4.10
pars.sigma_tilde = pars.minimal_sigma_tilde()
pars.q = pars.minimal_q() # Too many parameters. need fixed-point iteration...

print(pars.is_correct())
print(pars.is_secure(rough=False))
print(f"{pars}")
print(pars.as_init_string())

In [ ]:
# Trying out l_A = 3 with f=256, n=12.

pars = RBEParams(secpar=128, f=256, n=12, l_A=3, k=3, gbase=32768, q=35184372088832, sigma=2**4.000, sigma_tilde=2**33.849312, sigma_star=2**3.750000, D=14, B=60)

pars.gbase = floor(2**10)
pars.sigma = 2**3.65
pars.sigma_star = 2**3.2
pars.sigma_tilde = pars.minimal_sigma_tilde()
pars.q = pars.minimal_q() # Too many parameters. need fixed-point iteration...


print(pars.is_correct())
print(pars.is_secure(rough=True))
print(f"{pars}")
print(pars.as_init_string())